# 🤖 ROBOPILOT — CLOUD D3QN TRAINING & BENCHMARK PIPELINE
### Khung huấn luyện Deep Reinforcement Learning cho AMR Kho Vận thông minh

> **Designed by:** Tab 4 (@tab_4) — ML/RL Engineer & Training Architect  
> **Specification:** `docs/TRAINING_PIPELINE.md`, `docs/MODEL_LIFECYCLE.md`  
> **Hardware Target:** Google Colab GPU (T4 / V100 / A100)

Notebook này cho phép huấn luyện mô hình **Dueling Double Deep Q-Network (D3QN)** trên hạ tầng điện toán GPU đám mây, tự động đóng gói theo **Hợp đồng Kỹ thuật số 6 (Contract #6)** gồm `model.pt` và `metadata.json` để chuyển giao cho Backend/Frontend.

## 1. Thiết lập Môi trường & Clone Repository

In [ ]:
# 1. Clone repository Robopilot
!git clone https://github.com/HieuTran2901/Robopilot.git
%cd Robopilot

# 2. Cài đặt các thư viện cần thiết
!pip install gymnasium torch numpy matplotlib tqdm pytest

# 3. Kiểm tra GPU và PyTorch
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Hardware Accelerator: {device}")
if torch.cuda.is_available():
    print(f"[*] GPU Model: {torch.cuda.get_device_name(0)}")

## 2. Kiểm thử Môi trường Kho Vận (WarehouseGridEnv-v0)
Kiểm tra môi trường Gymnasium, cơ chế Potential-based Reward Shaping ($PBRS$) và tính tương thích.

In [ ]:
from src.ai_core.env.warehouse_env import WarehouseGridEnv
import matplotlib.pyplot as plt
import numpy as np

env = WarehouseGridEnv(map_file="maps/benchmark_easy_10x10.json")
obs, info = env.reset(seed=42)
print(f"Observation shape (7-dim raycast): {obs.shape}")
print(f"Sample observation: {obs}")

# Trực quan hóa bản đồ kho
plt.figure(figsize=(6, 6))
plt.imshow(env.grid, cmap="gray_r", origin="upper")
plt.plot(env.start_pos[0], env.start_pos[1], "go", markersize=14, label="Start Pos")
plt.plot(env.goal_pos[0], env.goal_pos[1], "r*", markersize=18, label="Goal Pos")
plt.title(f"RoboPilot Warehouse Map ({env.width}x{env.height})")
plt.grid(True, which="both", color="gray", linestyle="--", linewidth=0.5)
plt.legend()
plt.show()

## 3. Khởi chạy Baseline A* Search (Ground Truth)
A* cung cấp lộ trình lý thuyết tối ưu để đối sánh hiệu quả với các thuật toán RL.

In [ ]:
from src.ai_core.algorithms.astar import AStarPlanner

planner = AStarPlanner(env.grid_map)
astar_res = planner.plan(env.start_pos, env.goal_pos)

print(f"[*] A* Tìm thấy đường: {astar_res['found']}")
print(f"[*] Số bước tối ưu: {astar_res['length']} bước")
print(f"[*] Thời gian thực thi: {astar_res['execution_time_ms']:.3f} ms")
print(f"[*] Số node duyệt: {astar_res['nodes_expanded']}")

## 4. Huấn luyện Mô hình Dueling Double DQN (D3QN)
Huấn luyện mạng nơ-ron với:
- Vector quan sát tương đối $\mathbb{R}^7$
- Hàm mất mát Robust Huber Loss (Smooth L1)
- Polyak Soft Target Update ($\tau = 0.005$)
- Triệt tiêu Reward Hacking bằng PBRS $\Phi(s) = -\text{Manhattan}(s, \text{goal}) / (W + H - 2)$

In [ ]:
from src.ai_core.algorithms.d3qn import D3QNAgent
from tqdm import tqdm
import time

NUM_EPISODES = 500
BATCH_SIZE = 64
LR = 0.0005
SEED = 42

agent = D3QNAgent(
    input_dim=7,
    action_dim=4,
    learning_rate=LR,
    batch_size=BATCH_SIZE,
    buffer_capacity=50000,
    seed=SEED,
    device=str(device),
)

rewards_history = []
success_history = []
loss_history = []

pbar = tqdm(range(1, NUM_EPISODES + 1), desc="Huấn luyện D3QN")
for ep in pbar:
    obs, _ = env.reset(seed=SEED + ep)
    done = False
    truncated = False
    ep_reward = 0.0
    losses = []
    
    while not (done or truncated):
        action = agent.select_action(obs, evaluate=False)
        next_obs, reward, done, truncated, info = env.step(action)
        
        # Bootstrap invariant: chỉ terminal khi thành công hoặc va chạm thật
        terminal = done and not truncated
        agent.store_transition(obs, action, reward, next_obs, terminal)
        
        l = agent.train_step()
        if l is not None:
            losses.append(l)
            
        obs = next_obs
        ep_reward += reward
        
    agent.decay_epsilon()
    rewards_history.append(ep_reward)
    success_history.append(1 if info.get("is_success", False) else 0)
    if losses:
        loss_history.append(np.mean(losses))
        
    if ep % 25 == 0:
        recent_succ = np.mean(success_history[-25:]) * 100.0
        recent_rew = np.mean(rewards_history[-25:])
        pbar.set_postfix({"Success": f"{recent_succ:.1f}%", "Reward": f"{recent_rew:.1f}", "Eps": f"{agent.epsilon:.2f}"})

print("[*] Hoàn thành huấn luyện D3QN!")

## 5. Đồ thị Đường cong Học tập (Learning Curves)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Đồ thị Phần thưởng
window = 20
smooth_rewards = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
ax1.plot(rewards_history, alpha=0.3, color='steelblue', label='Raw Episode Reward')
ax1.plot(range(window-1, len(rewards_history)), smooth_rewards, color='navy', linewidth=2, label=f'Rolling Avg ({window} eps)')
ax1.set_title("Hội tụ Phần thưởng (Episode Reward Convergence)")
ax1.set_xlabel("Episode")
ax1.set_ylabel("Reward")
ax1.grid(True, linestyle="--", alpha=0.6)
ax1.legend()

# Đồ thị Tỷ lệ Thành công
smooth_succ = np.convolve(success_history, np.ones(window)/window, mode='valid') * 100.0
ax2.plot(range(window-1, len(success_history)), smooth_succ, color='forestgreen', linewidth=2, label=f'Success Rate ({window} eps)')
ax2.axhline(y=95.0, color='red', linestyle='--', label='Production Target (95%)')
ax2.set_title("Tỷ lệ Đến Đích Thành công (%)")
ax2.set_xlabel("Episode")
ax2.set_ylabel("Success Rate (%)")
ax2.set_ylim(-5, 105)
ax2.grid(True, linestyle="--", alpha=0.6)
ax2.legend()

plt.tight_layout()
plt.show()

## 6. Đóng gói Model Artifact theo Hợp đồng Kỹ thuật #6 & Tải về

In [ ]:
import shutil
from google.colab import files
from src.ai_core.algorithms.artifact import ModelArtifactManager

final_succ = float(np.mean(success_history[-100:]) * 100.0)
final_rew = float(np.mean(rewards_history[-100:]))

meta_overrides = {
    "metrics": {
        "success_rate_pct": final_succ,
        "mean_reward": final_rew,
        "episodes_trained": NUM_EPISODES,
    },
    "training_config": {
        "learning_rate": LR,
        "batch_size": BATCH_SIZE,
        "seed": SEED,
    }
}

artifact_dir = "models/d3qn_warehouse_v1"
exported_path = ModelArtifactManager.export_artifact(artifact_dir, agent, meta_overrides)
print(f"[*] Đã đóng gói Artifact tại: {exported_path}")

# Nén thành file zip để tải về
zip_filename = shutil.make_archive("d3qn_warehouse_v1", 'zip', artifact_dir)
print(f"[*] File nén sẵn sàng tải về: {zip_filename}")
files.download(zip_filename)
